# Stacked Ensemble Model

This notebook creates a **stacked ensemble** using predictions from:
1. **exp_8_hybrid_charcnn_mlp** - Hybrid CharCNN + MLP (Neural Network)
2. **exp_4_all_xgboost_optuna** - XGBoost with Optuna tuning
3. **exp_2_tfidf_ridge** - Ridge Classifier with TF-IDF

## Ensemble Architecture

**Level 0 (Base Models):**
- Load pre-trained models from experiment directories
- Generate out-of-fold predictions on train data (for meta-learner training)
- Generate predictions on test data

**Level 1 (Meta-Learner):**
- Logistic Regression trained on base model predictions
- Uses the same 5-fold CV structure as base models (random_state=42)

All models use the **same fold splits** (StratifiedKFold with random_state=42), ensuring proper alignment of out-of-fold predictions.

## Important Notes

**Hybrid Model Requirements:**
- The Keras hybrid model (exp_8) requires both character sequences and MLP features
- Loads: model, preprocessor (dict with 'preprocessor' and 'scaler'), and char tokenizer
- Preprocessing: text → char sequences (pad to 1000) + TF-IDF+structured → scaled features
- Model expects: `[char_input, mlp_input]`

In [1]:
use_drive = False

# uncomment if running on colab
# from google.colab import drive
# drive.mount('/content/drive')
# use_drive = True

In [2]:
import pandas as pd
import numpy as np
import pickle
import json
import os
from datetime import datetime

# Sklearn tools
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, log_loss
)

# Scipy for sigmoid conversion
from scipy.special import expit

# Deep learning (for Keras models)
import tensorflow as tf
from tensorflow import keras

# Optuna for hyperparameter tuning
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate
)

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Optuna version: {optuna.__version__}")

TensorFlow version: 2.20.0
Keras version: 3.12.0
Optuna version: 4.5.0


## 1. Configuration and Setup

In [3]:
if use_drive:
    DRIVE_PATH = '/content/drive/MyDrive/Kaggle'
    DATA_PATH = os.path.join(DRIVE_PATH, "data/")
    EXPERIMENT_PATH = os.path.join(DRIVE_PATH, "experiments/")
else:
    DATA_PATH = "data/"
    EXPERIMENT_PATH = "experiments/"

ENSEMBLE_NAME = "exp_10_stacked_ensemble"
ENSEMBLE_DIR = os.path.join(EXPERIMENT_PATH, ENSEMBLE_NAME)

# Create ensemble directory
os.makedirs(ENSEMBLE_DIR, exist_ok=True)
print(f"Ensemble directory: {ENSEMBLE_DIR}")

# Base model configurations
BASE_MODELS = {
    'xgboost_optuna': {
        'path': os.path.join(EXPERIMENT_PATH, 'exp_4_all_xgboost') if use_drive else 'experiments/exp_4_all_xgboost',
        'type': 'sklearn',
        'pipeline_prefix': 'pipeline_fold_',
        'pipeline_suffix': '_optuna.pkl'
    },
    'ridge': {
        'path': os.path.join(EXPERIMENT_PATH, 'exp_2_tfidf_ridge') if use_drive else 'experiments/exp_2_tfidf_ridge',
        'type': 'sklearn',
        'pipeline_prefix': 'pipeline_fold_',
        'pipeline_suffix': '.pkl'
    },
    'hybrid_charcnn_mlp': {
        'path': os.path.join(EXPERIMENT_PATH, 'exp_8_hybrid_charcnn_mlp') if use_drive else 'experiments/exp_8_hybrid_charcnn_mlp',
        'type': 'keras',
        'model_prefix': 'models/model_fold_',
        'preprocessor_prefix': 'models/preprocessor_fold_',
        'model_suffix': '.keras',
        'preprocessor_suffix': '.pkl',
        'tokenizer_file': 'tokenizer.json'  # Char tokenizer for CharCNN
    }
}

# Cross-validation setup (MUST match base models)
N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"\nData path: {DATA_PATH}")
print(f"Experiment path: {EXPERIMENT_PATH}")
print(f"Base models: {list(BASE_MODELS.keys())}")
print(f"Cross-validation: {N_FOLDS} folds, random_state={RANDOM_STATE}")

Ensemble directory: experiments/exp_10_stacked_ensemble

Data path: data/
Experiment path: experiments/
Base models: ['xgboost_optuna', 'ridge', 'hybrid_charcnn_mlp']
Cross-validation: 5 folds, random_state=42


## 2. Load Data

In [4]:
# Load training and test data
train = pd.read_csv(os.path.join(DATA_PATH, "train_w_struct.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test_w_struct.csv"))

# Deduplicate rows on text column
train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Extract target and ids
y = train['label'].values
test_ids = test['id'].values

print(f"\nTarget distribution:")
print(f"  Class 0: {np.sum(y == 0)} ({np.sum(y == 0) / len(y) * 100:.2f}%)")
print(f"  Class 1: {np.sum(y == 1)} ({np.sum(y == 1) / len(y) * 100:.2f}%)")

Train shape: (4212, 36)
Test shape: (1178, 35)

Target distribution:
  Class 0: 2733 (64.89%)
  Class 1: 1479 (35.11%)


## 3. Load Base Model Metrics

First, let's load and display the performance of each base model.

In [5]:
# Load metrics for each base model
base_model_metrics = {}

for model_name, config in BASE_MODELS.items():
    metrics_path = os.path.join(config['path'], 'metrics.json')
    if model_name == 'xgboost_optuna':
        metrics_path = os.path.join(config['path'], 'metrics_optuna.json')
    
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
        base_model_metrics[model_name] = metrics
        
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print(f"Type: {metrics['model_type']}")
    print(f"Vectorizer: {metrics['vectorizer']}")
    print(f"\nAverage Metrics (across {metrics['n_folds']} folds):")
    for metric, value in metrics['average_metrics'].items():
        if '_std' not in metric:
            std_value = metrics['average_metrics'].get(f"{metric}_std", 0)
            print(f"  {metric:12s}: {value:.6f} ± {std_value:.6f}")


Model: xgboost_optuna
Type: XGBoost
Vectorizer: TF-IDF+TruncatedSVD

Average Metrics (across 5 folds):
  accuracy    : 0.969654 ± 0.006380
  precision   : 0.963915 ± 0.009998
  recall      : 0.938463 ± 0.013117
  f1          : 0.950987 ± 0.010372
  roc_auc     : 0.995114 ± 0.001975
  log_loss    : 0.086210 ± 0.021418

Model: ridge
Type: Ridge Classifier
Vectorizer: TF-IDF

Average Metrics (across 5 folds):
  accuracy    : 0.974744 ± 0.006122
  precision   : 0.985008 ± 0.006408
  recall      : 0.933733 ± 0.016164
  f1          : 0.958639 ± 0.010280
  roc_auc     : 0.997613 ± 0.000646

Model: hybrid_charcnn_mlp
Type: Hybrid CharCNN + MLP (Keras)
Vectorizer: CharTokenizer + TF-IDF+TruncatedSVD

Average Metrics (across 5 folds):
  accuracy    : 0.970502 ± 0.005525
  precision   : 0.964657 ± 0.008873
  recall      : 0.940497 ± 0.012571
  f1          : 0.952393 ± 0.009078
  roc_auc     : 0.994832 ± 0.004155
  log_loss    : 0.082699 ± 0.026219


## 4. Generate Out-of-Fold Predictions (Train)

For the stacked ensemble, we need out-of-fold predictions from each base model on the training data. Since all models use the same fold splits, we can load predictions from each fold.

In [6]:
# Apply sklearn compatibility fix once at module level
import sys
from sklearn.compose import _column_transformer

if not hasattr(_column_transformer, '_RemainderColsList'):
    class _RemainderColsList(list):
        pass
    _column_transformer._RemainderColsList = _RemainderColsList
    sys.modules['sklearn.compose._column_transformer']._RemainderColsList = _RemainderColsList
    print("✓ Applied sklearn compatibility fix for _RemainderColsList")


def load_char_tokenizer(tokenizer_path):
    """Load character tokenizer from JSON (shared across all folds)."""
    with open(tokenizer_path, 'r') as f:
        tokenizer_data = json.load(f)
    
    from tensorflow.keras.preprocessing.text import Tokenizer
    char_tokenizer = Tokenizer(char_level=True, lower=True, filters='')
    char_tokenizer.word_index = tokenizer_data['word_index']
    
    return char_tokenizer


def load_keras_model_and_preprocessor(model_path, preprocessor_path):
    """Load a Keras model and its preprocessor (without tokenizer)."""
    # Load model (compile=False for faster loading since we only need inference)
    model = keras.models.load_model(model_path, compile=False)
    
    # Load preprocessor dict (contains 'preprocessor' and 'scaler')
    with open(preprocessor_path, 'rb') as f:
        preprocessor_dict = pickle.load(f)
    
    return model, preprocessor_dict


def load_sklearn_pipeline(pipeline_path):
    """Load a sklearn pipeline."""
    with open(pipeline_path, 'rb') as f:
        pipeline = pickle.load(f)
    return pipeline


def predict_keras_model(model, preprocessor_dict, char_tokenizer, data, maxlen=1000):
    """Make predictions using Keras hybrid model (CharCNN + MLP)."""
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    
    # Extract preprocessor and scaler
    preprocessor = preprocessor_dict['preprocessor']
    scaler = preprocessor_dict['scaler']
    
    # Process MLP features (TF-IDF + structured features)
    X_mlp_processed = preprocessor.transform(data)
    X_mlp_scaled = scaler.transform(X_mlp_processed)
    
    # Process CharCNN features (character sequences)
    texts = data['text'].astype(str).values
    X_char_seq = char_tokenizer.texts_to_sequences(texts)
    X_char_pad = pad_sequences(X_char_seq, maxlen=maxlen, padding='post', truncating='post')
    
    # Model expects [char_input, mlp_input]
    predictions = model.predict([X_char_pad, X_mlp_scaled], verbose=0).flatten()
    
    return predictions


def predict_sklearn_pipeline(pipeline, data, model_name=''):
    """Make predictions using sklearn pipeline."""
    # For Ridge model (text-only), extract just the text column
    # For XGBoost model (all features), use full dataframe
    if 'ridge' in model_name.lower():
        # Ridge expects only text data
        if isinstance(data, pd.DataFrame) and 'text' in data.columns:
            data_to_predict = data['text'].values
        else:
            raise ValueError(f"Ridge model requires DataFrame with 'text' column, got {type(data)}")
    else:
        # XGBoost and other models expect full dataframe
        data_to_predict = data.reset_index(drop=True) if isinstance(data, pd.DataFrame) else data
    
    # Check if pipeline has predict_proba (most classifiers)
    if hasattr(pipeline, 'predict_proba'):
        predictions = pipeline.predict_proba(data_to_predict)[:, 1]
    # Otherwise use decision_function (e.g., RidgeClassifier, SVM)
    elif hasattr(pipeline, 'decision_function'):
        # decision_function returns raw scores, need to convert to probabilities
        decision_scores = pipeline.decision_function(data_to_predict)
        predictions = expit(decision_scores)  # Sigmoid to convert to [0, 1]
    else:
        raise AttributeError(f"Pipeline {pipeline} has neither predict_proba nor decision_function")
    
    # Verify shape
    expected_len = len(data)
    if len(predictions) != expected_len:
        raise ValueError(f"Prediction length mismatch for {model_name}: expected {expected_len}, got {len(predictions)}")
    
    return predictions

✓ Applied sklearn compatibility fix for _RemainderColsList


In [ ]:
# Initialize arrays to store out-of-fold predictions (for meta-learner training)
oof_predictions = {
    'hybrid_charcnn_mlp': np.zeros(len(train)),
    'xgboost_optuna': np.zeros(len(train)),
    'ridge': np.zeros(len(train))
}

# Initialize arrays to store test predictions (averaged across folds)
test_predictions = {
    'hybrid_charcnn_mlp': np.zeros(len(test)),
    'xgboost_optuna': np.zeros(len(test)),
    'ridge': np.zeros(len(test))
}

print("Generating out-of-fold and test predictions from base models...")
print("="*60)

# Pre-load character tokenizer once (shared across all hybrid model folds)
char_tokenizer = None
for model_name, config in BASE_MODELS.items():
    if config['type'] == 'keras' and 'tokenizer_file' in config:
        tokenizer_path = os.path.join(config['path'], config['tokenizer_file'])
        print(f"\nLoading shared character tokenizer from {config['path']}...")
        char_tokenizer = load_char_tokenizer(tokenizer_path)
        print(f"✓ Tokenizer loaded (vocab size: {len(char_tokenizer.word_index)})")
        break

print()

# Iterate through folds
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train, y), 1):
    print(f"\nProcessing Fold {fold_idx}/{N_FOLDS}")
    print(f"  Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
    # Get validation data for this fold
    val_data = train.iloc[val_idx].drop(['id', 'label'], axis=1)
    test_data = test.drop(['id'], axis=1)
    
    # Process each base model
    for model_name, config in BASE_MODELS.items():
        print(f"  Loading {model_name}...", end=" ", flush=True)
        
        if config['type'] == 'keras':
            # Load Keras model
            model_path = os.path.join(
                config['path'], 
                f"{config['model_prefix']}{fold_idx}{config['model_suffix']}"
            )
            preprocessor_path = os.path.join(
                config['path'],
                f"{config['preprocessor_prefix']}{fold_idx}{config['preprocessor_suffix']}"
            )
            
            # Load model and preprocessor (tokenizer already loaded)
            model, preprocessor_dict = load_keras_model_and_preprocessor(
                model_path, preprocessor_path
            )
            
            # Generate predictions
            val_preds = predict_keras_model(model, preprocessor_dict, char_tokenizer, val_data)
            test_preds = predict_keras_model(model, preprocessor_dict, char_tokenizer, test_data)
            
            # Clean up to free memory
            del model
            keras.backend.clear_session()
            
        elif config['type'] == 'sklearn':
            # Load sklearn pipeline
            pipeline_path = os.path.join(
                config['path'],
                f"{config['pipeline_prefix']}{fold_idx}{config['pipeline_suffix']}"
            )
            pipeline = load_sklearn_pipeline(pipeline_path)
            
            # Generate predictions (pass model_name to handle different input types)
            val_preds = predict_sklearn_pipeline(pipeline, val_data, model_name)
            test_preds = predict_sklearn_pipeline(pipeline, test_data, model_name)
        
        # Store out-of-fold predictions
        oof_predictions[model_name][val_idx] = val_preds
        
        # Accumulate test predictions (will average later)
        test_predictions[model_name] += test_preds / N_FOLDS
        
        print(f"✓ (val: {len(val_preds)}, test: {len(test_preds)})")

print("\n" + "="*60)
print("Base model prediction generation complete!")
print("\nOut-of-fold prediction shapes:")
for model_name, preds in oof_predictions.items():
    print(f"  {model_name:20s}: {preds.shape}")
print("\nTest prediction shapes:")
for model_name, preds in test_predictions.items():
    print(f"  {model_name:20s}: {preds.shape}")

Generating out-of-fold and test predictions from base models...

Loading shared character tokenizer from experiments/exp_8_hybrid_charcnn_mlp...
✓ Tokenizer loaded (vocab size: 772)


Processing Fold 1/5
  Train size: 3369, Val size: 843
  Loading xgboost_optuna... ✓ (val: 843, test: 1178)
  Loading ridge... ✓ (val: 843, test: 1178)
  Loading hybrid_charcnn_mlp... 

## 5. Evaluate Base Model Performance

Let's check the performance of each base model on the training data using their out-of-fold predictions.

In [ ]:
print("Base Model Performance (Out-of-Fold):")
print("="*60)

base_model_oof_metrics = {}

for model_name, preds in oof_predictions.items():
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y, (preds >= 0.5).astype(int)),
        'precision': precision_score(y, (preds >= 0.5).astype(int)),
        'recall': recall_score(y, (preds >= 0.5).astype(int)),
        'f1': f1_score(y, (preds >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y, preds),
        'log_loss': log_loss(y, preds)
    }
    
    base_model_oof_metrics[model_name] = metrics
    
    print(f"\n{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric:12s}: {value:.6f}")

print("\n" + "="*60)

## 6. Train Meta-Learner (Stacking)

Now we train a Logistic Regression meta-learner on the out-of-fold predictions from the base models.

In [ ]:
# Create feature matrix from out-of-fold predictions
X_meta_train = np.column_stack([
    oof_predictions['hybrid_charcnn_mlp'],
    oof_predictions['xgboost_optuna'],
    oof_predictions['ridge']
])

print(f"Meta-learner training data shape: {X_meta_train.shape}")
print(f"Target shape: {y.shape}")

# Meta-learner parameters
meta_learner_params = {
    'C': 1.0,
    'max_iter': 1000,
    'random_state': RANDOM_STATE,
    'solver': 'lbfgs'
}

print(f"\nMeta-learner parameters: {meta_learner_params}")

## 7. Evaluate Ensemble Performance

In [ ]:
# Train meta-learner with cross-validation to evaluate performance
meta_oof_predictions = np.zeros(len(train))
meta_test_predictions = np.zeros(len(test))

meta_fold_metrics = []

print("\nTraining meta-learner with 5-fold CV...")
print("="*60)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, y), 1):
    print(f"\nFold {fold_idx}/{N_FOLDS}")
    
    # Split meta-features
    X_meta_fold_train = X_meta_train[train_idx]
    X_meta_fold_val = X_meta_train[val_idx]
    y_fold_train = y[train_idx]
    y_fold_val = y[val_idx]
    
    # Train meta-learner
    meta_learner = LogisticRegression(**meta_learner_params)
    meta_learner.fit(X_meta_fold_train, y_fold_train)
    
    # Predict on validation fold
    val_preds = meta_learner.predict_proba(X_meta_fold_val)[:, 1]
    meta_oof_predictions[val_idx] = val_preds
    
    # Predict on test (will average later)
    X_meta_test = np.column_stack([
        test_predictions['hybrid_charcnn_mlp'],
        test_predictions['xgboost_optuna'],
        test_predictions['ridge']
    ])
    test_preds = meta_learner.predict_proba(X_meta_test)[:, 1]
    meta_test_predictions += test_preds / N_FOLDS
    
    # Calculate fold metrics
    fold_metrics = {
        'fold': fold_idx,
        'accuracy': accuracy_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'precision': precision_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'recall': recall_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'f1': f1_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y_fold_val, val_preds),
        'log_loss': log_loss(y_fold_val, val_preds),
        'train_size': len(train_idx),
        'val_size': len(val_idx)
    }
    
    meta_fold_metrics.append(fold_metrics)
    
    print(f"  ROC-AUC: {fold_metrics['roc_auc']:.6f}")
    print(f"  Log Loss: {fold_metrics['log_loss']:.6f}")
    print(f"  F1 Score: {fold_metrics['f1']:.6f}")
    
    # Save meta-learner for this fold
    meta_learner_path = os.path.join(ENSEMBLE_DIR, f'meta_learner_fold_{fold_idx}.pkl')
    with open(meta_learner_path, 'wb') as f:
        pickle.dump(meta_learner, f)

print("\n" + "="*60)
print("Meta-learner training complete!")

## 7a. Optuna Hyperparameter Tuning for Meta-Learner

Now let's use Optuna to find the best hyperparameters for the meta-learner.

In [ ]:
def objective(trial):
    """Optuna objective function for meta-learner hyperparameter tuning."""
    
    # Define hyperparameter search space
    params = {
        'C': trial.suggest_float('C', 0.001, 10.0, log=True),
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'saga']),
        'max_iter': trial.suggest_int('max_iter', 100, 2000, step=100),
        'random_state': RANDOM_STATE
    }
    
    # Add penalty parameter for compatible solvers
    if params['solver'] in ['liblinear', 'saga']:
        params['penalty'] = trial.suggest_categorical('penalty', ['l1', 'l2'])
    else:
        params['penalty'] = 'l2'
    
    # Cross-validation scores
    fold_scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, y), 1):
        # Split meta-features
        X_meta_fold_train = X_meta_train[train_idx]
        X_meta_fold_val = X_meta_train[val_idx]
        y_fold_train = y[train_idx]
        y_fold_val = y[val_idx]
        
        # Train meta-learner
        meta_learner = LogisticRegression(**params)
        meta_learner.fit(X_meta_fold_train, y_fold_train)
        
        # Predict on validation fold
        val_preds = meta_learner.predict_proba(X_meta_fold_val)[:, 1]
        
        # Calculate ROC-AUC
        fold_score = roc_auc_score(y_fold_val, val_preds)
        fold_scores.append(fold_score)
    
    # Return average ROC-AUC across folds
    return np.mean(fold_scores)


# Create Optuna study
print("Starting Optuna hyperparameter optimization...")
print("="*60)

study = optuna.create_study(
    direction='maximize',
    study_name='meta_learner_optimization',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

# Run optimization
study.optimize(objective, n_trials=100, show_progress_bar=True)

# Display best results
print("\n" + "="*60)
print("Optuna Optimization Complete!")
print("="*60)
print(f"\nBest trial:")
print(f"  Value (ROC-AUC): {study.best_value:.6f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key:12s}: {value}")

# Save study
study_path = os.path.join(ENSEMBLE_DIR, 'optuna_study.pkl')
with open(study_path, 'wb') as f:
    pickle.dump(study, f)
print(f"\nStudy saved to: {study_path}")

## 7b. Visualize Optuna Results

In [ ]:
# Create directory for Optuna plots
optuna_plots_dir = os.path.join(ENSEMBLE_DIR, 'optuna_plots')
os.makedirs(optuna_plots_dir, exist_ok=True)

# Plot optimization history
fig1 = plot_optimization_history(study)
fig1.write_html(os.path.join(optuna_plots_dir, 'optimization_history.html'))
print(f"Optimization history saved to: {optuna_plots_dir}/optimization_history.html")

# Plot parameter importances
fig2 = plot_param_importances(study)
fig2.write_html(os.path.join(optuna_plots_dir, 'param_importances.html'))
print(f"Parameter importances saved to: {optuna_plots_dir}/param_importances.html")

# Plot parallel coordinate
fig3 = plot_parallel_coordinate(study)
fig3.write_html(os.path.join(optuna_plots_dir, 'parallel_coordinate.html'))
print(f"Parallel coordinate saved to: {optuna_plots_dir}/parallel_coordinate.html")

print("\nOptuna visualizations saved successfully!")
print("Open the HTML files in a browser to view the plots.")

## 7c. Train Tuned Meta-Learner with Best Parameters

In [ ]:
# Get best parameters from Optuna study
best_params = study.best_params.copy()
best_params['random_state'] = RANDOM_STATE

print("Training tuned meta-learner with best parameters...")
print("="*60)
print(f"Best parameters: {best_params}")
print()

# Train tuned meta-learner with cross-validation
meta_oof_predictions_optuna = np.zeros(len(train))
meta_test_predictions_optuna = np.zeros(len(test))

meta_fold_metrics_optuna = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, y), 1):
    print(f"\nFold {fold_idx}/{N_FOLDS}")
    
    # Split meta-features
    X_meta_fold_train = X_meta_train[train_idx]
    X_meta_fold_val = X_meta_train[val_idx]
    y_fold_train = y[train_idx]
    y_fold_val = y[val_idx]
    
    # Train tuned meta-learner
    meta_learner_optuna = LogisticRegression(**best_params)
    meta_learner_optuna.fit(X_meta_fold_train, y_fold_train)
    
    # Predict on validation fold
    val_preds = meta_learner_optuna.predict_proba(X_meta_fold_val)[:, 1]
    meta_oof_predictions_optuna[val_idx] = val_preds
    
    # Predict on test (will average later)
    X_meta_test = np.column_stack([
        test_predictions['hybrid_charcnn_mlp'],
        test_predictions['xgboost_optuna'],
        test_predictions['ridge']
    ])
    test_preds = meta_learner_optuna.predict_proba(X_meta_test)[:, 1]
    meta_test_predictions_optuna += test_preds / N_FOLDS
    
    # Calculate fold metrics
    fold_metrics = {
        'fold': fold_idx,
        'accuracy': accuracy_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'precision': precision_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'recall': recall_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'f1': f1_score(y_fold_val, (val_preds >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y_fold_val, val_preds),
        'log_loss': log_loss(y_fold_val, val_preds),
        'train_size': len(train_idx),
        'val_size': len(val_idx)
    }
    
    meta_fold_metrics_optuna.append(fold_metrics)
    
    print(f"  ROC-AUC: {fold_metrics['roc_auc']:.6f}")
    print(f"  Log Loss: {fold_metrics['log_loss']:.6f}")
    print(f"  F1 Score: {fold_metrics['f1']:.6f}")
    
    # Save tuned meta-learner for this fold
    meta_learner_path = os.path.join(ENSEMBLE_DIR, f'meta_learner_fold_{fold_idx}_optuna.pkl')
    with open(meta_learner_path, 'wb') as f:
        pickle.dump(meta_learner_optuna, f)

print("\n" + "="*60)
print("Tuned meta-learner training complete!")

# Calculate overall tuned meta-learner metrics
meta_metrics_optuna = {
    'accuracy': accuracy_score(y, (meta_oof_predictions_optuna >= 0.5).astype(int)),
    'precision': precision_score(y, (meta_oof_predictions_optuna >= 0.5).astype(int)),
    'recall': recall_score(y, (meta_oof_predictions_optuna >= 0.5).astype(int)),
    'f1': f1_score(y, (meta_oof_predictions_optuna >= 0.5).astype(int)),
    'roc_auc': roc_auc_score(y, meta_oof_predictions_optuna),
    'log_loss': log_loss(y, meta_oof_predictions_optuna)
}

print("\nTuned Meta-Learner Performance (Out-of-Fold):")
print("="*60)
for metric, value in meta_metrics_optuna.items():
    print(f"  {metric:12s}: {value:.6f}")

# Calculate average and std across folds
avg_meta_metrics_optuna = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    values = [fold[metric] for fold in meta_fold_metrics_optuna]
    avg_meta_metrics_optuna[metric] = np.mean(values)
    avg_meta_metrics_optuna[f"{metric}_std"] = np.std(values)

print("\nTuned Meta-Learner Average Fold Performance:")
print("="*60)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    mean = avg_meta_metrics_optuna[metric]
    std = avg_meta_metrics_optuna[f"{metric}_std"]
    print(f"  {metric:12s}: {mean:.6f} ± {std:.6f}")

In [ ]:
# Calculate overall meta-learner metrics
meta_metrics = {
    'accuracy': accuracy_score(y, (meta_oof_predictions >= 0.5).astype(int)),
    'precision': precision_score(y, (meta_oof_predictions >= 0.5).astype(int)),
    'recall': recall_score(y, (meta_oof_predictions >= 0.5).astype(int)),
    'f1': f1_score(y, (meta_oof_predictions >= 0.5).astype(int)),
    'roc_auc': roc_auc_score(y, meta_oof_predictions),
    'log_loss': log_loss(y, meta_oof_predictions)
}

print("\nMeta-Learner Performance (Out-of-Fold):")
print("="*60)
for metric, value in meta_metrics.items():
    print(f"  {metric:12s}: {value:.6f}")

# Calculate average and std across folds
avg_meta_metrics = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    values = [fold[metric] for fold in meta_fold_metrics]
    avg_meta_metrics[metric] = np.mean(values)
    avg_meta_metrics[f"{metric}_std"] = np.std(values)

print("\nMeta-Learner Average Fold Performance:")
print("="*60)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    mean = avg_meta_metrics[metric]
    std = avg_meta_metrics[f"{metric}_std"]
    print(f"  {metric:12s}: {mean:.6f} ± {std:.6f}")

## 8. Simple Average Ensemble (Baseline)

For comparison, let's also create a simple average ensemble of the base model predictions.

In [ ]:
# Simple average of base model predictions
avg_oof_predictions = np.mean([
    oof_predictions['hybrid_charcnn_mlp'],
    oof_predictions['xgboost_optuna'],
    oof_predictions['ridge']
], axis=0)

avg_test_predictions = np.mean([
    test_predictions['hybrid_charcnn_mlp'],
    test_predictions['xgboost_optuna'],
    test_predictions['ridge']
], axis=0)

# Calculate metrics for average ensemble
avg_metrics = {
    'accuracy': accuracy_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'precision': precision_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'recall': recall_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'f1': f1_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'roc_auc': roc_auc_score(y, avg_oof_predictions),
    'log_loss': log_loss(y, avg_oof_predictions)
}

print("\nSimple Average Ensemble Performance:")
print("="*60)
for metric, value in avg_metrics.items():
    print(f"  {metric:12s}: {value:.6f}")

## 9. Performance Comparison

In [ ]:
# Create comparison dataframe
comparison_data = []

# Base models
for model_name, metrics in base_model_oof_metrics.items():
    comparison_data.append({
        'Model': model_name,
        'Type': 'Base Model',
        'ROC-AUC': metrics['roc_auc'],
        'Log Loss': metrics['log_loss'],
        'F1 Score': metrics['f1'],
        'Accuracy': metrics['accuracy']
    })

# Simple average
comparison_data.append({
    'Model': 'Simple Average',
    'Type': 'Ensemble',
    'ROC-AUC': avg_metrics['roc_auc'],
    'Log Loss': avg_metrics['log_loss'],
    'F1 Score': avg_metrics['f1'],
    'Accuracy': avg_metrics['accuracy']
})

# Stacked ensemble (meta-learner - untuned)
comparison_data.append({
    'Model': 'Stacked (Untuned)',
    'Type': 'Ensemble',
    'ROC-AUC': meta_metrics['roc_auc'],
    'Log Loss': meta_metrics['log_loss'],
    'F1 Score': meta_metrics['f1'],
    'Accuracy': meta_metrics['accuracy']
})

# Stacked ensemble (meta-learner - Optuna tuned)
comparison_data.append({
    'Model': 'Stacked (Optuna)',
    'Type': 'Ensemble',
    'ROC-AUC': meta_metrics_optuna['roc_auc'],
    'Log Loss': meta_metrics_optuna['log_loss'],
    'F1 Score': meta_metrics_optuna['f1'],
    'Accuracy': meta_metrics_optuna['accuracy']
})

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False)

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Identify best model
best_model_idx = comparison_df['ROC-AUC'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_roc_auc = comparison_df.loc[best_model_idx, 'ROC-AUC']

print(f"\nBest Model: {best_model} (ROC-AUC: {best_roc_auc:.6f})")

## 10. Generate Submission Files

In [ ]:
# Create submission for simple average ensemble
submission_avg = pd.DataFrame({
    'id': test_ids,
    'probability': avg_test_predictions
})
submission_path_avg = os.path.join(ENSEMBLE_DIR, f'{ENSEMBLE_NAME}_avg_submission.csv')
submission_avg.to_csv(submission_path_avg, index=False)
print(f"Simple average submission saved to: {submission_path_avg}")

# Create submission for stacked ensemble (untuned)
submission_stacked = pd.DataFrame({
    'id': test_ids,
    'probability': meta_test_predictions
})
submission_path_stacked = os.path.join(ENSEMBLE_DIR, f'{ENSEMBLE_NAME}_submission.csv')
submission_stacked.to_csv(submission_path_stacked, index=False)
print(f"Stacked ensemble (untuned) submission saved to: {submission_path_stacked}")

# Create submission for stacked ensemble (Optuna tuned)
submission_stacked_optuna = pd.DataFrame({
    'id': test_ids,
    'probability': meta_test_predictions_optuna
})
submission_path_stacked_optuna = os.path.join(ENSEMBLE_DIR, f'{ENSEMBLE_NAME}_submission_optuna.csv')
submission_stacked_optuna.to_csv(submission_path_stacked_optuna, index=False)
print(f"Stacked ensemble (Optuna) submission saved to: {submission_path_stacked_optuna}")

# Display sample
print("\nSample predictions (Optuna tuned):")
print(submission_stacked_optuna.head(10))

## 11. Save Metrics (Following Model Tracking Guide)

In [ ]:
# Identify best fold for untuned model
best_fold_idx = max(range(len(meta_fold_metrics)), 
                    key=lambda i: meta_fold_metrics[i]['roc_auc'])
best_fold = best_fold_idx + 1

# Identify best fold for Optuna tuned model
best_fold_idx_optuna = max(range(len(meta_fold_metrics_optuna)), 
                           key=lambda i: meta_fold_metrics_optuna[i]['roc_auc'])
best_fold_optuna = best_fold_idx_optuna + 1

# Create comprehensive metrics JSON
ensemble_metrics = {
    "experiment_name": ENSEMBLE_NAME,
    "model_type": "Stacked Ensemble (Logistic Regression Meta-Learner)",
    "base_models": [
        {
            "name": "exp_8_hybrid_charcnn_mlp",
            "type": "Hybrid CharCNN + MLP",
            "oof_roc_auc": base_model_oof_metrics['hybrid_charcnn_mlp']['roc_auc']
        },
        {
            "name": "exp_4_all_xgboost_optuna", 
            "type": "XGBoost (Optuna)",
            "oof_roc_auc": base_model_oof_metrics['xgboost_optuna']['roc_auc']
        },
        {
            "name": "exp_2_tfidf_ridge",
            "type": "Ridge Classifier",
            "oof_roc_auc": base_model_oof_metrics['ridge']['roc_auc']
        }
    ],
    "meta_learner": "Logistic Regression",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_folds": N_FOLDS,
    "random_state": RANDOM_STATE,
    
    # Untuned meta-learner
    "untuned": {
        "meta_learner_params": meta_learner_params,
        "best_fold": best_fold,
        "average_metrics": avg_meta_metrics,
        "fold_metrics": meta_fold_metrics,
        "overall_metrics": meta_metrics
    },
    
    # Optuna tuned meta-learner
    "optuna": {
        "n_trials": len(study.trials),
        "best_params": best_params,
        "best_value": study.best_value,
        "study_path": "optuna_study.pkl",
        "best_fold": best_fold_optuna,
        "average_metrics": avg_meta_metrics_optuna,
        "fold_metrics": meta_fold_metrics_optuna,
        "overall_metrics": meta_metrics_optuna
    },
    
    "comparison": {
        "simple_average": avg_metrics,
        "stacked_untuned": meta_metrics,
        "stacked_optuna": meta_metrics_optuna,
        "improvement_optuna_over_avg": {
            "roc_auc": meta_metrics_optuna['roc_auc'] - avg_metrics['roc_auc'],
            "log_loss": avg_metrics['log_loss'] - meta_metrics_optuna['log_loss']
        },
        "improvement_optuna_over_untuned": {
            "roc_auc": meta_metrics_optuna['roc_auc'] - meta_metrics['roc_auc'],
            "log_loss": meta_metrics['log_loss'] - meta_metrics_optuna['log_loss']
        }
    }
}

# Save metrics
metrics_path = os.path.join(ENSEMBLE_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(ensemble_metrics, f, indent=2)

print(f"Metrics saved to: {metrics_path}")

## 12. Save Out-of-Fold Predictions

Save the out-of-fold predictions for potential future use.

In [ ]:
# Save out-of-fold predictions from base models and ensemble methods
oof_predictions_df = pd.DataFrame({
    'id': train['id'].values,
    'label': y,
    'hybrid_charcnn_mlp': oof_predictions['hybrid_charcnn_mlp'],
    'xgboost_optuna': oof_predictions['xgboost_optuna'],
    'ridge': oof_predictions['ridge'],
    'simple_average': avg_oof_predictions,
    'stacked_untuned': meta_oof_predictions,
    'stacked_optuna': meta_oof_predictions_optuna
})

oof_path = os.path.join(ENSEMBLE_DIR, 'oof_predictions.csv')
oof_predictions_df.to_csv(oof_path, index=False)
print(f"Out-of-fold predictions saved to: {oof_path}")

# Display sample
print("\nSample OOF predictions:")
print(oof_predictions_df.head(10))

## 13. Final Summary

In [ ]:
print("\n" + "="*80)
print("ENSEMBLE MODEL SUMMARY")
print("="*80)

print("\nBase Models:")
for i, model_info in enumerate(ensemble_metrics['base_models'], 1):
    print(f"  {i}. {model_info['name']}")
    print(f"     Type: {model_info['type']}")
    print(f"     OOF ROC-AUC: {model_info['oof_roc_auc']:.6f}")

print("\n" + "-"*80)
print("ENSEMBLE METHODS:")
print("-"*80)

print("\n1. Simple Average Ensemble:")
print(f"   ROC-AUC:  {avg_metrics['roc_auc']:.6f}")
print(f"   Log Loss: {avg_metrics['log_loss']:.6f}")

print("\n2. Stacked Ensemble (Untuned Meta-Learner):")
print(f"   Algorithm: {ensemble_metrics['meta_learner']}")
print(f"   Parameters: {meta_learner_params}")
print(f"   ROC-AUC:  {meta_metrics['roc_auc']:.6f}")
print(f"   Log Loss: {meta_metrics['log_loss']:.6f}")
print(f"   Improvement over Simple Average: {ensemble_metrics['comparison']['improvement_optuna_over_avg']['roc_auc']:+.6f} (ROC-AUC)")

print("\n3. Stacked Ensemble (Optuna Tuned Meta-Learner):")
print(f"   Algorithm: {ensemble_metrics['meta_learner']}")
print(f"   Best Parameters: {best_params}")
print(f"   Optuna Trials: {len(study.trials)}")
print(f"   ROC-AUC:  {meta_metrics_optuna['roc_auc']:.6f}")
print(f"   Log Loss: {meta_metrics_optuna['log_loss']:.6f}")
print(f"   Improvement over Simple Average: {ensemble_metrics['comparison']['improvement_optuna_over_avg']['roc_auc']:+.6f} (ROC-AUC)")
print(f"   Improvement over Untuned: {ensemble_metrics['comparison']['improvement_optuna_over_untuned']['roc_auc']:+.6f} (ROC-AUC)")

print("\n" + "-"*80)
print("SAVED ARTIFACTS:")
print("-"*80)
print(f"  1. Metrics: {metrics_path}")
print(f"  2. Simple Average Submission: {submission_path_avg}")
print(f"  3. Stacked (Untuned) Submission: {submission_path_stacked}")
print(f"  4. Stacked (Optuna) Submission: {submission_path_stacked_optuna}")
print(f"  5. OOF Predictions: {oof_path}")
print(f"  6. Untuned Meta-learner models: {ENSEMBLE_DIR}/meta_learner_fold_*.pkl")
print(f"  7. Optuna Meta-learner models: {ENSEMBLE_DIR}/meta_learner_fold_*_optuna.pkl")
print(f"  8. Optuna Study: {study_path}")
print(f"  9. Optuna Plots: {optuna_plots_dir}/")

print("\n" + "="*80)
print("ENSEMBLE MODEL COMPLETE!")
print("="*80)